# 01 — Load & Filter

Loads NYC PLUTO and Census block group boundaries, filters both down to Bushwick (Community District 304), and builds the point geometry needed for the spatial join in the next notebook.

Outputs (written to `cache/`): `bw_pluto.geojson`, `bg_kings.geojson`.


In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


In [2]:
# Filter PLUTO to Bushwick (Community District 304) and build point geometry.
DATA_DIR = '../../../data'

df_pluto = pd.read_csv(f'{DATA_DIR}/pluto.csv')
df_bw = df_pluto[df_pluto['community board'] == 304.0].copy()
df_bw = df_bw.dropna(subset=['latitude', 'longitude'])

geometry = [Point(xy) for xy in zip(df_bw['longitude'], df_bw['latitude'])]
gdf_pluto = gpd.GeoDataFrame(df_bw, geometry=geometry, crs='EPSG:4326')

print(gdf_pluto.shape)

/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_8362/2134130712.py:4: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: unitsres, 8: unitstotal, 9: lotfront, 10: lotdepth, 11: bldgfront, 12: bldgdepth, 13: irrlotcode, 14: histdist, 15: landmark, 16: condono, 17: edesignum, 18: dcpedited, 19: mihopt1, 20: mihopt2, 21: mihopt3, 22: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pluto = pd.read_csv(f'{DATA_DIR}/pluto.csv')


(11154, 109)


In [3]:
# Download Census block group boundaries for Kings County (Brooklyn).
# Public TIGER/Line geometry file, no API key needed. ZCTAs don't nest under
# state for the ACS data API, but this is the raw geometry file, not that API —
# it's fine to filter by COUNTYFP after loading.
url = 'https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_36_bg.zip'
gdf_bg_ny = gpd.read_file(url)
gdf_bg = gdf_bg_ny[gdf_bg_ny['COUNTYFP'] == '047'].copy()

gdf_pluto = gdf_pluto.to_crs(gdf_bg.crs)

print(gdf_bg.shape, gdf_pluto.crs == gdf_bg.crs)

(2156, 13) True


In [4]:
gdf_pluto.to_file('cache/bw_pluto.geojson', driver='GeoJSON')
gdf_bg.to_file('cache/bg_kings.geojson', driver='GeoJSON')
print('cached bw_pluto.geojson and bg_kings.geojson')

cached bw_pluto.geojson and bg_kings.geojson
